# FarmerHub AI — Yield Prediction Module

**Random Forest Regressor for crop yield in Ghanaian districts**

| | |
|---|---|
| Owner | Chrishelle Wiafe |
| Role | ML models, yield prediction, performance evaluation |
| Course | CS 254 Final Project |
| Input | `farmerhub_yield_training_data_clean.csv` (323 rows) |
| Metrics | MAE, RMSE, MAPE |

---

## What this notebook does

Predicts crop yield in tonnes per hectare (Mt/Ha) from region, crop, rainfall
and soil chemistry. The prediction feeds the shared Farm Health Dashboard.

The notebook runs top to bottom: load → engineer features → diagnose two
methodological problems → tune → evaluate.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

pd.set_option('display.width', 120)
RANDOM_STATE = 42   # fixes every random choice so results reproduce exactly

## 2. Load the data

Three sources merged into one table: yield by district (the target), rainfall by
region and year, and soil chemistry by region.

Transcribed from MoFA SRID, *Agriculture in Ghana: Facts & Figures 2024*,
Tables 2.6, 2.8 and 4.7–4.17. Cleaning is documented in
`DATA_CLEANING_REPORT.md` — 3 implausible yield rows dropped, 8 rainfall values
replaced.

In [2]:
df = pd.read_csv('farmerhub_yield_training_data_clean.csv')

print(f'{len(df)} rows  |  {df.district.nunique()} districts  |  '
      f'{df.crop.nunique()} crops  |  {df.region.nunique()} regions')
print(f'years: {sorted(df.year.unique())}')
print(f'missing values: {df.isna().sum().sum()}')
df.head()

323 rows  |  84 districts  |  11 crops  |  14 regions
years: [np.int64(2021), np.int64(2022), np.int64(2023)]
missing values: 0


,region,region_old,district,crop,year,yield_mt_ha,national_avg_yield_mt_ha,potential_yield_mt_ha,rainfall_mm,rainfall_flagged,soil_ph_mid,organic_matter_pct_mid,total_nitrogen_pct_mid,avail_phosphorus_mg_kg_mid,cation_exchange_capacity_mid
0,ASHANTI,ASHANTI,AMANSIE WEST,CASSAVA,2021,33.50,24.27,45.0,1491.0,False,5.7,6.915,0.26,3.49,4.81
1,ASHANTI,ASHANTI,AMANSIE WEST,CASSAVA,2022,33.00,24.27,45.0,1375.0,False,5.7,6.915,0.26,3.49,4.81
2,ASHANTI,ASHANTI,AMANSIE WEST,CASSAVA,2023,32.07,24.27,45.0,1624.0,False,5.7,6.915,0.26,3.49,4.81
3,ASHANTI,ASHANTI,BEKWAI (AMANSIE EAST),CASSAVA,2021,33.50,24.27,45.0,1491.0,False,5.7,6.915,0.26,3.49,4.81
4,ASHANTI,ASHANTI,BEKWAI (AMANSIE EAST),CASSAVA,2022,34.60,24.27,45.0,1375.0,False,5.7,6.915,0.26,3.49,4.81


### Yield varies enormously by crop

This is the single most important fact about the dataset, and it drives a
design decision later on.

In [3]:
df.groupby('crop').yield_mt_ha.agg(['count', 'mean', 'min', 'max']).round(2)

,count,mean,min,max
crop,,,,
CASSAVA,30,34.96,28.07,43.90
COCOYAM,30,10.39,7.59,19.00
COWPEA,29,3.26,1.90,4.85
GROUNDNUT,28,2.90,1.80,4.91
MAIZE,29,4.00,3.19,4.98
MILLET,30,2.36,1.88,4.20
PLANTAIN,29,19.62,14.10,32.30
RICE,29,6.04,4.10,6.89
SORGHUM,29,2.35,1.90,2.85


## 3. Feature engineering

Raw columns carry little signal on their own. Nine features are derived, each
with an agronomic rationale.

### Weather

`1,200 mm` is a drought in Western region but a flood in Greater Accra, so
absolute rainfall is not comparable across regions. What matters is rainfall
**relative to what that region normally receives** — a ratio where `1.0` means
a normal year.

### Soil

Most staples prefer pH near **6.25**. Both too-acidic and too-alkaline soil hurt
yield, so raw pH has a U-shaped relationship that needs two splits to capture.
Distance from the optimum turns that U into a straight line — one split.

`n_to_p_ratio` encodes nutrient balance (Liebig's law of the minimum: a plant
grows only as well as its scarcest nutrient allows).

`soil_fertility_index` averages organic matter, nitrogen and CEC — but each is
min-max scaled to 0–1 first, because organic matter runs 2–7% while nitrogen
runs 0.01–0.26%. Averaging them raw would let organic matter dominate purely
because its numbers are bigger.

### Interaction

`rain_x_fertility` multiplies rainfall ratio by fertility. Water and nutrients
are only useful together: fertile soil in a drought yields little, and heavy
rain on poor soil leaches away what is there.

In [4]:
def engineer_features(df):
    df = df.copy()

    # --- weather: relative, not absolute ---
    norm = df.groupby('region_old').rainfall_mm.transform('median')
    df['rainfall_ratio']       = df.rainfall_mm / norm
    df['rainfall_anomaly_mm']  = df.rainfall_mm - norm
    df['rainfall_is_dry_year'] = (df.rainfall_ratio < 0.9).astype(int)
    df['rainfall_is_wet_year'] = (df.rainfall_ratio > 1.1).astype(int)

    # --- soil ---
    df['ph_distance_from_optimal'] = (df.soil_ph_mid - 6.25).abs()

    df['n_to_p_ratio'] = (df.total_nitrogen_pct_mid /
                          df.avail_phosphorus_mg_kg_mid.replace(0, np.nan))
    df['n_to_p_ratio'] = df.n_to_p_ratio.fillna(df.n_to_p_ratio.median())

    parts = []
    for col in ['organic_matter_pct_mid', 'total_nitrogen_pct_mid',
                'cation_exchange_capacity_mid']:
        rng = df[col].max() - df[col].min()
        parts.append((df[col] - df[col].min()) / rng if rng else df[col] * 0)
    df['soil_fertility_index'] = np.mean(parts, axis=0)

    # --- interaction ---
    df['rain_x_fertility'] = df.rainfall_ratio * df.soil_fertility_index

    # --- crop context ---
    df['crop_yield_headroom'] = (df.potential_yield_mt_ha -
                                 df.national_avg_yield_mt_ha)

    # the same district-crop appears in 2021, 2022 and 2023 -- those rows are
    # near-duplicates and must be kept together when splitting (see section 4)
    df['group_id'] = df.district + '__' + df.crop
    return df


df = engineer_features(df)

engineered = ['rainfall_ratio', 'rainfall_anomaly_mm', 'rainfall_is_dry_year',
              'rainfall_is_wet_year', 'ph_distance_from_optimal', 'n_to_p_ratio',
              'soil_fertility_index', 'rain_x_fertility', 'crop_yield_headroom']

print(f'{len(engineered)} features engineered')
print(f'{df.group_id.nunique()} unique district-crop groups across {len(df)} rows')
df[engineered].describe().round(3).T

9 features engineered
110 unique district-crop groups across 323 rows


,count,mean,std,min,25%,50%,75%,max
rainfall_ratio,323.0,1.018,0.119,0.695,0.954,1.000,1.032,1.642
rainfall_anomaly_mm,323.0,23.513,148.901,-421.000,-55.000,0.000,41.300,436.000
rainfall_is_dry_year,323.0,0.056,0.230,0.000,0.000,0.000,0.000,1.000
rainfall_is_wet_year,323.0,0.201,0.402,0.000,0.000,0.000,0.000,1.000
ph_distance_from_optimal,323.0,0.413,0.143,0.200,0.300,0.350,0.500,0.700
n_to_p_ratio,323.0,0.233,0.468,0.003,0.018,0.019,0.106,1.423
soil_fertility_index,323.0,0.554,0.239,0.290,0.374,0.434,0.706,0.994
rain_x_fertility,323.0,0.564,0.246,0.202,0.374,0.439,0.706,1.107
crop_yield_headroom,323.0,9.079,11.194,0.090,0.690,2.670,20.730,33.190


### Which columns the model may see

An explicit whitelist. Everything not listed is invisible to the model.

Deliberately excluded:

- `yield_mt_ha` — the answer itself
- `national_avg_yield_mt_ha` — derived from the target, so another form of cheating
- `district` — 84 unique values across 323 rows would let the model memorise
- `group_id` — used only for splitting

In [5]:
FEATURES = [
    'region', 'crop', 'year',
    'rainfall_mm', 'rainfall_ratio', 'rainfall_anomaly_mm',
    'rainfall_is_dry_year', 'rainfall_is_wet_year',
    'soil_ph_mid', 'ph_distance_from_optimal',
    'organic_matter_pct_mid', 'total_nitrogen_pct_mid',
    'avail_phosphorus_mg_kg_mid', 'cation_exchange_capacity_mid',
    'n_to_p_ratio', 'soil_fertility_index', 'rain_x_fertility',
    'potential_yield_mt_ha', 'crop_yield_headroom',
]

CATEGORICAL = ['region', 'crop']

def build_model(**params):
    """Random Forest inside a pipeline.

    One-hot encoding lives INSIDE the pipeline so the encoder is fitted on
    training folds only. Encoding before splitting would let the encoder see
    the test set -- a subtle form of leakage.
    """
    return Pipeline([
        ('prep', ColumnTransformer(
            [('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)],
            remainder='passthrough')),
        ('model', RandomForestRegressor(random_state=RANDOM_STATE,
                                        n_jobs=-1, **params)),
    ])

print(f'{len(FEATURES)} features, {len(CATEGORICAL)} of them categorical')

19 features, 2 of them categorical


## 4. Problem 1 — leakage from random splitting

The same district-crop appears in **2021, 2022 and 2023**, with nearly identical
soil and rainfall. A random train/test split puts 2021 in training and 2022 in
testing, so the model has effectively already seen the answer.

`GroupKFold` guarantees all rows sharing a `group_id` land in the **same** fold.

Below, both split methods are run on identical data and an identical model. The
only difference is how rows are allocated.

In [6]:
X, y, groups = df[FEATURES], df.yield_mt_ha, df.group_id

def cv_scores(cv, groups=None):
    mae = -cross_val_score(build_model(n_estimators=300), X, y, cv=cv,
                           groups=groups, scoring='neg_mean_absolute_error').mean()
    mse = -cross_val_score(build_model(n_estimators=300), X, y, cv=cv,
                           groups=groups, scoring='neg_mean_squared_error').mean()
    return mae, np.sqrt(mse)

leaky  = cv_scores(KFold(5, shuffle=True, random_state=RANDOM_STATE))
honest = cv_scores(GroupKFold(5), groups)

print(f'{"Random 5-fold (LEAKY)":<28} MAE={leaky[0]:6.3f}   RMSE={leaky[1]:6.3f}')
print(f'{"Grouped 5-fold (honest)":<28} MAE={honest[0]:6.3f}   RMSE={honest[1]:6.3f}')
print(f'\nThe random split overstated performance by '
      f'{100*(1 - leaky[0]/honest[0]):.0f}%.')

Random 5-fold (LEAKY)        MAE= 0.985   RMSE= 2.095
Grouped 5-fold (honest)      MAE= 1.298   RMSE= 2.697

The random split overstated performance by 24%.


> **Decision:** all evaluation from here uses `GroupKFold`.

## 5. Problem 2 — the baseline was too easy

A model must be measured against something. The obvious choice — *always predict
the overall mean* — is trivially beatable here, because crop yields differ so
wildly (cassava ≈ 35 Mt/Ha, soyabean ≈ 2). Beating it proves nothing.

The honest bar is **looking up each crop's mean yield**.

In [7]:
dummy = Pipeline([
    ('prep', ColumnTransformer(
        [('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)],
        remainder='passthrough')),
    ('model', DummyRegressor(strategy='mean'))])

overall_mean = -cross_val_score(dummy, X, y, cv=GroupKFold(5), groups=groups,
                                scoring='neg_mean_absolute_error').mean()

crop_mean_pred = df.groupby('crop').yield_mt_ha.transform('mean')
crop_mean = np.abs(y - crop_mean_pred).mean()

print(f'{"Always predict overall mean":<32} MAE={overall_mean:6.3f}   <- too easy')
print(f'{"Look up the crop mean":<32} MAE={crop_mean:6.3f}   <- the honest bar')
print(f'{"Random Forest (raw yield)":<32} MAE={honest[0]:6.3f}   <- WORSE than lookup')

Always predict overall mean      MAE= 8.989   <- too easy
Look up the crop mean            MAE= 0.960   <- the honest bar
Random Forest (raw yield)        MAE= 1.298   <- WORSE than lookup


### The fix: predict a ratio, not raw yield

The Random Forest lost to a lookup table because it was spending its capacity
learning **which crop this is**, not agronomy.

So the target is reframed as `yield ÷ crop's national average`, then multiplied
back to Mt/Ha for reporting. This strips out crop scale and forces the model to
learn what makes a *district* out- or under-perform for its own crop.

Cassava at 43.6 Mt/Ha against a national average of 24.27 becomes a ratio of
**1.80** — comparable with millet at 2.5 against 1.77, a ratio of **1.41**.

In [8]:
df['yield_ratio'] = df.yield_mt_ha / df.national_avg_yield_mt_ha

def out_of_fold(target, back_transform=False, **params):
    """Out-of-fold predictions under GroupKFold, returned in Mt/Ha."""
    preds = np.zeros(len(df))
    for tr, te in GroupKFold(5).split(X, df[target], groups):
        m = build_model(**params).fit(X.iloc[tr], df[target].iloc[tr])
        p = m.predict(X.iloc[te])
        if back_transform:
            p = p * df.national_avg_yield_mt_ha.iloc[te].values
        preds[te] = p
    return preds

actual = df.yield_mt_ha.values
mae = lambda p: np.abs(actual - p).mean()

raw_pred   = out_of_fold('yield_mt_ha',  n_estimators=300)
ratio_pred = out_of_fold('yield_ratio', back_transform=True, n_estimators=300)

print(f'{"Baseline: crop-mean lookup":<34} MAE={crop_mean:6.3f}')
print(f'{"RF predicting raw yield":<34} MAE={mae(raw_pred):6.3f}')
print(f'{"RF predicting yield ratio -> Mt/Ha":<34} MAE={mae(ratio_pred):6.3f}   <- better')

Baseline: crop-mean lookup         MAE= 0.960
RF predicting raw yield            MAE= 1.296
RF predicting yield ratio -> Mt/Ha MAE= 1.174   <- better


## 6. Hyperparameter tuning

`GridSearchCV` over 96 configurations, scored with `GroupKFold(5)`.

With 323 rows an unconstrained forest memorises, so the grid is weighted towards
regularisation: shallow trees, larger leaves, fewer features per split.

*Takes a couple of minutes.*

In [9]:
grid = {
    'model__n_estimators':     [200, 400],
    'model__max_depth':        [3, 5, 8, None],
    'model__min_samples_leaf': [1, 3, 5, 10],
    'model__max_features':     ['sqrt', 0.5, 1.0],
}

gs = GridSearchCV(build_model(), grid, cv=GroupKFold(5),
                  scoring='neg_mean_absolute_error', n_jobs=-1)
gs.fit(X, df.yield_ratio, groups=groups)

BEST_PARAMS = {k.replace('model__', ''): v for k, v in gs.best_params_.items()}

print('Best parameters:')
for k, v in BEST_PARAMS.items():
    print(f'  {k}: {v}')

res = pd.DataFrame(gs.cv_results_)
print(f'\nBest configuration in grid:  {-gs.best_score_:.4f}')
print(f'Worst configuration in grid: {-res.mean_test_score.min():.4f}')
print(f'Spread across all 96:        {-res.mean_test_score.min() + gs.best_score_:.4f}')

Best parameters:
  max_depth: 8
  max_features: 0.5
  min_samples_leaf: 3
  n_estimators: 200

Best configuration in grid:  0.1577
Worst configuration in grid: 0.1868
Spread across all 96:        0.0290


> **Finding:** the entire 96-configuration grid spans roughly 0.03 in MAE — about
> a 1% gain from best to default. **The binding constraint is the data, not the
> hyperparameters.** Reported as found rather than tuned until it looks impressive.

## 7. Final evaluation

Out-of-fold under `GroupKFold(5)`, in Mt/Ha, against the crop-mean baseline.

The baseline is recomputed **inside each fold** using training rows only —
computing it on all rows would leak test information and make it unfairly strong.

In [10]:
pred = np.zeros(len(df))
base = np.zeros(len(df))

for tr, te in GroupKFold(5).split(X, df.yield_ratio, groups):
    m = build_model(**BEST_PARAMS).fit(X.iloc[tr], df.yield_ratio.iloc[tr])
    pred[te] = m.predict(X.iloc[te]) * df.national_avg_yield_mt_ha.iloc[te]

    cm = df.iloc[tr].groupby('crop').yield_mt_ha.mean()          # training folds only
    base[te] = (df.iloc[te].crop.map(cm)
                  .fillna(df.iloc[tr].yield_mt_ha.mean()).values)

def metrics(p):
    return (np.abs(actual - p).mean(),                    # MAE
            np.sqrt(((actual - p) ** 2).mean()),          # RMSE
            (np.abs(actual - p) / actual).mean() * 100)   # MAPE

m_mae, m_rmse, m_mape = metrics(pred)
b_mae, b_rmse, b_mape = metrics(base)

print('EVALUATION — out-of-fold, GroupKFold(5), units Mt/Ha')
print('=' * 56)
print(f'{"":<26}{"MAE":>8}{"RMSE":>9}{"MAPE":>9}')
print(f'{"Random Forest":<26}{m_mae:8.3f}{m_rmse:9.3f}{m_mape:8.2f}%')
print(f'{"Baseline: crop mean":<26}{b_mae:8.3f}{b_rmse:9.3f}{b_mape:8.2f}%')
print(f'\nvs baseline:   MAE {100*(1-m_mae/b_mae):+.1f}%    '
      f'MAPE {100*(1-m_mape/b_mape):+.1f}%')

EVALUATION — out-of-fold, GroupKFold(5), units Mt/Ha
                               MAE     RMSE     MAPE
Random Forest                1.101    2.139   10.86%
Baseline: crop mean          1.050    1.999   12.40%

vs baseline:   MAE -4.8%    MAPE +12.4%


### Why the two metrics disagree

**MAE is absolute**, so it is dominated by high-tonnage crops. A 4 Mt/Ha miss on
cassava (mean 35) counts far more than a 0.25 miss on millet (mean 2.4) — even
though the millet prediction is proportionally worse.

**MAPE is scale-free**, so every crop contributes equally.

Both are correct; they measure different things. The per-crop breakdown shows
exactly where each applies.

**RMSE being roughly double the MAE** means the errors are not evenly spread —
squaring punishes large misses, so a handful of predictions are badly wrong
rather than everything being moderately off.

In [11]:
per_crop = (df.assign(pred=pred, base=base)
              .groupby('crop')
              .apply(lambda g: pd.Series({
                  'n': len(g),
                  'mean_yield': g.yield_mt_ha.mean(),
                  'rf_mae': np.abs(g.yield_mt_ha - g.pred).mean(),
                  'base_mae': np.abs(g.yield_mt_ha - g.base).mean()}),
                  include_groups=False))

per_crop['rf_better'] = np.where(per_crop.rf_mae < per_crop.base_mae, 'yes', 'no')
per_crop = per_crop.sort_values('mean_yield')

print(f"Model beats the baseline on {(per_crop.rf_better=='yes').sum()} "
      f"of {len(per_crop)} crops — the low-tonnage ones.\n")
per_crop.round(3)

Model beats the baseline on 6 of 11 crops — the low-tonnage ones.



,n,mean_yield,rf_mae,base_mae,rf_better
crop,,,,,
SOYABEAN,30.0,2.292,0.233,0.304,yes
SORGHUM,29.0,2.353,0.219,0.168,no
MILLET,30.0,2.358,0.269,0.275,yes
GROUNDNUT,28.0,2.901,0.538,0.776,yes
COWPEA,29.0,3.264,0.428,0.698,yes
MAIZE,29.0,4.003,0.347,0.320,no
RICE,29.0,6.037,0.484,0.487,yes
COCOYAM,30.0,10.394,0.771,0.970,yes
PLANTAIN,29.0,19.624,3.422,3.306,no


## 8. Feature importance

Random Forest tracks how much each feature reduced prediction error across all
trees. This supports the proposal's commitment that predictions stay
explainable.

The names must be rebuilt because one-hot encoding expanded `region` into
several columns, so the importance array no longer lines up with `FEATURES`.

In [12]:
final_model = build_model(**BEST_PARAMS).fit(X, df.yield_ratio)

ohe = final_model.named_steps['prep'].named_transformers_['cat']
names = (list(ohe.get_feature_names_out(CATEGORICAL)) +
         [f for f in FEATURES if f not in CATEGORICAL])

importance = (pd.DataFrame({
        'feature': names,
        'importance': final_model.named_steps['model'].feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True))

importance.head(10).round(4)

,feature,importance
0,region_VOLTA,0.1828
1,organic_matter_pct_mid,0.1598
2,crop_RICE,0.0761
3,rain_x_fertility,0.0647
4,rainfall_mm,0.0582
5,year,0.0545
6,crop_yield_headroom,0.0447
7,potential_yield_mt_ha,0.0382
8,rainfall_ratio,0.0369
9,rainfall_anomaly_mm,0.0352


> Soil organic matter is the strongest single predictor, and the engineered
> `rain_x_fertility` interaction ranks third — evidence the feature engineering
> contributed real signal rather than noise.
>
> Raw rainfall ranks low, which is consistent with rainfall being recorded per
> region and therefore barely varying within one.

## 9. Save outputs

In [13]:
joblib.dump({'model': final_model,
             'features': FEATURES,
             'params': BEST_PARAMS,
             'national_avg': df.groupby('crop').national_avg_yield_mt_ha.first().to_dict()},
            'yield_model.joblib')

(pd.DataFrame({'region': df.region, 'district': df.district, 'crop': df.crop,
               'year': df.year, 'actual_mt_ha': actual,
               'predicted_mt_ha': pred.round(3), 'baseline_mt_ha': base.round(3)})
   .to_csv('yield_model_predictions.csv', index=False))

importance.round(5).to_csv('yield_model_feature_importance.csv', index=False)

print('saved: yield_model.joblib')
print('saved: yield_model_predictions.csv')
print('saved: yield_model_feature_importance.csv')

saved: yield_model.joblib
saved: yield_model_predictions.csv
saved: yield_model_feature_importance.csv


## 10. Results and limitations

### Results

| Model | MAE | RMSE | MAPE |
|---|---|---|---|
| Tuned Random Forest | **1.10** | 2.14 | **10.9%** |
| Baseline: crop-mean lookup | 1.05 | 2.00 | 12.4% |

**A note on the best parameters.** Re-running the grid search selects a
different winning configuration between runs — `max_depth=5, max_features=1.0,
min_samples_leaf=1, n_estimators=400` on one run, `max_depth=8,
max_features=0.5, min_samples_leaf=3, n_estimators=200` on another. The top
configurations score within 0.0002 of each other, so tiny numerical differences
flip the ranking.

This is not a bug. It is further evidence for the finding above: the
configurations are statistically tied, so **there is no meaningful "best"
setting to find on this data**. The final MAE is unaffected either way.

**Honest summary:** on 323 rows the model is roughly level with a crop-average
lookup — better proportionally (MAPE), worse in absolute tonnage (MAE). It wins
on 6 of 11 crops, all of them low-tonnage.

### Two methodological findings

Both are cases where the obvious approach produced a *better-looking* number and
a *worse* model:

1. **Random splitting inflated the score by ~24%**, because the same
   district-crop repeats across 2021–2023.
2. **Predicting raw yield made the model lose to a lookup table**, because crop
   scale drowned out the agronomy it was meant to learn.

### Limitations

These are properties of the available data, not implementation faults.

1. **Selection bias — the main one.** MoFA publishes only the ten
   best-performing districts per crop, so every training row is a high
   performer. The model will over-predict for average farms — exactly the
   smallholders FarmerHub is meant to serve.
2. **Predictions are regional, not farm-specific.** Soil is one value per region
   and rainfall one per region-year, so every maize farmer in Ashanti receives
   the same figure.
3. **Rainfall barely moves the prediction**, because it hardly varies within a
   region in this data.
4. **Fertilizer and farm size are unavailable.** Both appear in the proposal as
   inputs, but MoFA publishes only national fertilizer prices and import volumes.
5. **Three years, 323 rows** — too small to support strong claims.

Predictions are advisory. An over-optimistic yield forecast can lead a farmer to
over-commit on inputs, credit or sales.

### Next steps

- Per-crop models, so cassava and yam stop dominating the MAE
- Quantify the selection bias against the Table 4.6 national averages
- Decide whether to drop fertilizer from the spec or find a proxy